# feax4d — Fibre-path / G-code Verification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Naruki-Ichihara/feax4d/blob/main/examples/colab_verify.ipynb)

A quick, **optimisation-free** sandbox to verify the feax4d manufacturing
pipeline: build a simple analytic fibre layout (with svgpathtools), turn it into
a polymer-filled Fibrifier (9T Labs) G-code with a configurable layer stack, and
view the 3D print paths interactively.

Edit the **Configure** cell (Section 1) to change the plate size, fibre pattern,
layer stack and print parameters, then run the rest.

This notebook does not run the FE solver, so setup is light (~1–2 min).

## Install

`feax4d` is installed from GitHub (with `--no-deps`); we add its runtime deps
(`shapely`, `tsp-solver2`, `svgpathtools`) and the `feax` backend.  No
SuiteSparse / cuDSS build is needed here because no optimisation is run.

In [ ]:
!apt-get update
!apt-get install -y libglu1 libxcursor-dev libxft2 libxinerama1
!pip install -q "feax @ git+https://github.com/Naruki-Ichihara/feax.git"
!pip install -q shapely tsp-solver2 svgpathtools nlopt gmsh
!pip install -q --no-deps git+https://github.com/Naruki-Ichihara/feax4d.git

In [ ]:
import feax4d
print("feax4d", feax4d.__version__)

## 1. Configure — edit size & parameters here

Everything you'd normally tweak is in this one cell.

In [ ]:
# ===== Geometry & fibre pattern =====
WIDTH_MM   = 100.0      # plate size (fibres run along the LONGER axis)
HEIGHT_MM  = 50.0
PITCH_MM   = 2.0        # fibre spacing across the short axis [mm]
KIND       = "snake"    # "snake" (one continuous serpentine) | "lines" | "wave"

# ===== Layer stack =====
POLYMER_BASE_LAYERS     = 2      # pure-polymer layers below the fibre stack
FIBRE_LAYERS            = 10     # number of fibre layers
POLYMER_TOP_LAYERS      = 2      # pure-polymer layers above
POLYMER_IN_FIBRE_LAYERS = False  # False = fibre-only middle layers (polymer only in caps)
FIBER_CUT               = False  # False = continuous fibre (climb Z between layers, no cut)

# ===== Polymer infill =====
INFILL_ANGLE = 90.0     # infill direction [deg] (90 = across the fibres)
INFILL_PITCH = 0.8      # polymer line spacing [mm]
CONNECTION_THRESHOLD = 4.0   # merge fibre path ends within this [mm]

# ===== Print parameters =====
POLYMER_LAYER_HEIGHT = 0.15   # mm — cap / polymer-matrix layer thickness
FIBER_LAYER_HEIGHT   = 0.10   # mm — fibre layer thickness (set independently)
OFFSET_X, OFFSET_Y = 175.0, 135.0      # machine offsets [mm]
BED_TEMP   = 90         # °C
CF_TEMP    = 220        # fibre nozzle temperature [°C]
PL_TEMP    = 230        # polymer nozzle temperature [°C]
CF_FEEDRATE = 600       # fibre feedrate [mm/min]

## 2. Generate test fibre paths + Fibrifier g-code

In [ ]:
from pathlib import Path
import feax4d

out = Path("output_verify"); out.mkdir(exist_ok=True)
svg = out / "fibre_paths_layer0.svg"
feax4d.make_test_fibre_svg(svg, width=WIDTH_MM, height=HEIGHT_MM, pitch=PITCH_MM, kind=KIND)

# Print parameters (from the Configure cell).
params = feax4d.FibrifierParams()
params.layer_height = POLYMER_LAYER_HEIGHT   # fallback / base
params.offset_x, params.offset_y = OFFSET_X, OFFSET_Y
params.temperature.cf_print_temp = CF_TEMP
params.temperature.pl_print_temp = PL_TEMP
params.temperature.bed_temperature = BED_TEMP
params.speed.cf_print_feedrate = CF_FEEDRATE

result = feax4d.fibre_paths_to_gcode(
    [str(svg)],
    output_gcode=str(out / "verify.gcode"),
    params=params,
    polymer_fill=True,
    infill_angle=INFILL_ANGLE,
    infill_pitch=INFILL_PITCH,
    polymer_base_layers=POLYMER_BASE_LAYERS,
    layer_print_layers=FIBRE_LAYERS,
    polymer_top_layers=POLYMER_TOP_LAYERS,
    polymer_in_fiber_layers=POLYMER_IN_FIBRE_LAYERS,
    polymer_layer_height=POLYMER_LAYER_HEIGHT,
    fiber_layer_height=FIBER_LAYER_HEIGHT,
    fiber_cut=FIBER_CUT,
    connection_threshold=CONNECTION_THRESHOLD,
)
stack = "".join("F" if ld["fiber"] else "P" for ld in result["layers"])
print("\nlayer stack :", stack)
print("print layers:", result["n_layers"],
      "| fibre paths:", result["n_fiber_paths"],
      "| total fibre: %.0f mm" % result["total_fiber_mm"])
print("g-code      :", result["gcode_path"])

## 3. Interactive 3D print-path view

Drag to rotate, scroll to zoom.  Fibre = orange, polymer infill = green; the Z
axis is exaggerated by ``LAYER_GAP`` so the stack is legible.

In [ ]:
# Layer spacing here is a VIEWER-ONLY setting — it does not affect the
# g-code (which uses params.layer_height).  Raise it to spread layers apart.
LAYER_GAP = 6.0
fig = feax4d.plot_print_paths_plotly(result, layer_gap=LAYER_GAP)
fig.show()

## 4. Inspect / download the g-code

In [ ]:
with open(result["gcode_path"]) as f:
    print("".join(f.readlines()[:40]))

# Download the g-code (Colab only).
try:
    from google.colab import files
    files.download(result["gcode_path"])
except Exception:
    pass

## Notes

- **Continuous fibre** (`FIBER_CUT = False`): the fibre is never severed; between
  fibre layers it climbs Z at the same point (alternate layers are reversed so
  the climb is a clean vertical step). Consecutive fibre layers share tool T0 —
  no redundant tool change.
- **Layer stack**: `POLYMER_BASE_LAYERS` + `FIBRE_LAYERS` + `POLYMER_TOP_LAYERS`
  physical layers. With `POLYMER_IN_FIBRE_LAYERS = False` the middle layers are
  fibre-only (polymer lives only in the caps).
- Swap `KIND` to `"lines"` (straight) or `"wave"` (curved) to test other layouts.

Repository: <https://github.com/Naruki-Ichihara/feax4d>